# 📈 RetailPulse: análisis RFM de clientes (Online Retail II)

**Objetivo:** segmentar a los clientes de una tienda online según *cuándo* compraron por última vez (Recencia), *cuántas veces* compran (Frecuencia) y *cuánto* gastan (Monto), para decidir a quién premiar, fidelizar o reactivar.

**Cómo usar este notebook en Google Colab**
1. Sube `online_retail_II.xlsx` al panel de archivos 📁 (si descargaste un `.zip`, descomprímelo antes).
2. Ejecuta las celdas en orden (**Entorno de ejecución → Ejecutar todas**). La lectura del Excel puede tardar 1 o 2 minutos.

**Fuente de datos:** [Online Retail II, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii). Moneda: libras esterlinas (£).

## Bloque 1: Importar las librerías
Cargamos las herramientas para trabajar con tablas; sin ellas no podemos analizar las ventas.

In [ ]:
# Importamos pandas, la librería principal para trabajar con tablas de datos
import pandas as pd

# Importamos numpy, que nos ayuda con operaciones numéricas (lo usaremos poco)
import numpy as np

# Le pedimos a pandas que muestre los números con 2 decimales, para leerlos mejor
pd.options.display.float_format = '{:,.2f}'.format

## Bloque 2: Cargar los datos y ver un resumen
Antes de analizar ventas hay que saber qué tan completos y confiables son los datos.

In [ ]:
# Leemos TODAS las hojas del Excel (sheet_name=None); el resultado es un diccionario de tablas
hojas = pd.read_excel("online_retail_II.xlsx", sheet_name=None)

# Mostramos los nombres de las hojas para confirmar que se leyeron bien
print("Hojas encontradas:", list(hojas.keys()))

# Unimos todas las hojas una debajo de otra en una sola tabla; ignore_index reinicia la numeración de filas
df = pd.concat(hojas.values(), ignore_index=True)

# Imprimimos un título para ordenar la salida
print("\n===== RESUMEN INICIAL =====")

# shape nos da (filas, columnas); mostramos cada número por separado
print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")

# Mostramos el tipo de dato de cada columna (texto, número, fecha...)
print("\nTipos de datos:")
print(df.dtypes)

# isnull() marca los vacíos y sum() los cuenta por columna
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Mostramos las primeras 5 filas para "ver" cómo luce la tabla
df.head()

## Bloque 3: Limpiar los datos
Quitamos registros que no son compras reales de clientes identificados (ventas sin cliente, devoluciones, errores y pedidos atípicos); si no, distorsionan quién es un buen cliente.

> **Nota sobre los pedidos atípicos:** hay 2 líneas de más de 70,000 unidades en una sola compra (£168 mil y £77 mil). Son pedidos que se registraron y se anularon el mismo día. La anulación (factura con "C") ya la eliminamos, pero la compra original quedaba, inflando el Top de productos y los segmentos de dos clientes. Por eso se excluyen.

In [ ]:
# Guardamos cuántas filas teníamos al inicio para poder comparar después
filas_inicio = len(df)

# Hacemos una copia para no modificar la tabla original por accidente
limpio = df.copy()

# Nos quedamos solo con filas que SÍ tienen Customer ID (notna = "no es nulo")
limpio = limpio[limpio["Customer ID"].notna()]

# Convertimos Invoice a texto para poder revisar con qué letra empieza
limpio["Invoice"] = limpio["Invoice"].astype(str)

# Quitamos las facturas canceladas: las que empiezan con "C" (el ~ significa "lo contrario")
limpio = limpio[~limpio["Invoice"].str.startswith("C")]

# Nos quedamos solo con cantidades mayores a 0 (una cantidad 0 o negativa no es una venta)
limpio = limpio[limpio["Quantity"] > 0]

# Nos quedamos solo con precios mayores a 0 (precio 0 o negativo suele ser un ajuste o error)
limpio = limpio[limpio["Price"] > 0]

# Quitamos pedidos atípicos de más de 50,000 unidades en una línea (se registraron y anularon el mismo día)
limpio = limpio[limpio["Quantity"] < 50000]

# Convertimos Customer ID a número entero (venía como decimal, ej. 13085.0)
limpio["Customer ID"] = limpio["Customer ID"].astype(int)

# Reiniciamos la numeración de filas para que quede ordenada tras borrar
limpio = limpio.reset_index(drop=True)

# Mostramos cuántas filas había, cuántas quedaron y qué porcentaje se eliminó
print(f"Filas antes: {filas_inicio:,}")
print(f"Filas después: {len(limpio):,}")
print(f"Eliminado: {(1 - len(limpio)/filas_inicio):.1%}")

## Bloque 4: Crear la columna Ingreso
El negocio se mide en dinero, no solo en unidades: calculamos cuánto dejó cada línea de venta.

In [ ]:
# Multiplicamos cantidad por precio unitario para obtener el dinero de cada línea
limpio["Ingreso"] = limpio["Quantity"] * limpio["Price"]

# Mostramos el ingreso total del periodo como referencia general del negocio
print(f"Ingreso total del periodo: {limpio['Ingreso'].sum():,.2f}")

# Mostramos algunas filas con las columnas clave para verificar que el cálculo es correcto
limpio[["Invoice", "Quantity", "Price", "Ingreso"]].head()

## Bloque 5: Calcular Recencia, Frecuencia y Monto (RFM)
Pasamos de miles de transacciones a una fila por cliente, que es la unidad sobre la que se toman decisiones de marketing.

In [ ]:
# Buscamos la última fecha de compra del dataset
ultima_fecha = limpio["InvoiceDate"].max()

# La fecha de referencia es un día después de la última fecha (así nadie tiene recencia 0)
fecha_referencia = ultima_fecha + pd.Timedelta(days=1)

# Mostramos las dos fechas para entender desde dónde medimos
print("Última fecha del dataset:", ultima_fecha)
print("Fecha de referencia:", fecha_referencia)

# Agrupamos todas las compras por cliente y calculamos tres métricas a la vez
rfm = limpio.groupby("Customer ID").agg(
    # Recencia: fecha de referencia menos la última compra del cliente, expresado en días
    Recencia=("InvoiceDate", lambda fechas: (fecha_referencia - fechas.max()).days),
    # Frecuencia: contamos facturas ÚNICAS (una factura puede tener muchas líneas)
    Frecuencia=("Invoice", "nunique"),
    # Monto: sumamos todo el dinero que gastó el cliente
    Monto=("Ingreso", "sum")
).reset_index()  # reset_index convierte Customer ID de nuevo en una columna normal

# Mostramos cuántos clientes tenemos en total
print(f"\nClientes únicos: {len(rfm):,}")

# Mostramos estadísticas básicas (mínimo, promedio, máximo...) de las tres métricas
print(rfm[["Recencia", "Frecuencia", "Monto"]].describe())

# Mostramos las primeras filas
rfm.head()

## Bloque 6: Asignar puntajes de 1 a 5
Convertimos cada métrica en una escala simple (5 = mejor) comparando a cada cliente con los demás, para poder combinar días, compras y dinero.

In [ ]:
# Recencia: menos días = mejor cliente, así que los puntajes van al revés (5 = más reciente)
# qcut divide a los clientes en 5 grupos del mismo tamaño (quintiles)
rfm["R"] = pd.qcut(rfm["Recencia"], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frecuencia: más compras = mejor. Usamos rank(method="first") porque hay MUCHOS clientes con 1 sola compra
# y qcut fallaría con valores repetidos; rank los desempata para poder dividir en 5 grupos
rfm["F"] = pd.qcut(rfm["Frecuencia"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Monto: más dinero = mejor, así que 5 = quienes más gastan
rfm["M"] = pd.qcut(rfm["Monto"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Creamos un puntaje combinado de Frecuencia y Monto (su promedio), porque ambos miden "valor"
rfm["FM"] = (rfm["F"] + rfm["M"]) / 2

# Mostramos algunas filas para ver los puntajes asignados
rfm.head(10)

## Bloque 7: Crear los segmentos de negocio
Traducimos los puntajes a nombres que cualquiera entiende, para decidir una acción distinta con cada grupo.

| Segmento | Regla | Acción sugerida |
|---|---|---|
| Campeones | R ≥ 4 y FM ≥ 4 | Premiar y dar trato preferente |
| Clientes Leales | R ≥ 3 y FM ≥ 3 | Fidelizar y venta cruzada |
| Potenciales | El resto (recientes, poco historial) | Incentivar la recompra |
| En Riesgo | R ≤ 2 y FM > 2 | Campaña de reactivación |
| Perdidos | R ≤ 2 y FM ≤ 2 | Último intento de bajo costo |

In [ ]:
# Definimos una función que recibe una fila (un cliente) y devuelve su segmento
def asignar_segmento(fila):
    # Guardamos los puntajes en variables con nombres cortos para leer mejor
    r = fila["R"]
    fm = fila["FM"]

    # Campeones: compraron hace poco Y compran seguido o gastan mucho
    if r >= 4 and fm >= 4:
        return "Campeones"
    # Perdidos: hace mucho que no compran Y además compraron poco o gastaron poco
    elif r <= 2 and fm <= 2:
        return "Perdidos"
    # En Riesgo: hace tiempo que no compran, PERO antes eran buenos clientes (valor medio o alto)
    elif r <= 2 and fm > 2:
        return "En Riesgo"
    # Clientes Leales: compran con buena regularidad y valor, con recencia media o buena
    elif r >= 3 and fm >= 3:
        return "Clientes Leales"
    # Potenciales: todo lo demás; clientes recientes con poco historial que podrían crecer
    else:
        return "Potenciales"

# Aplicamos la función a cada fila (axis=1 significa "fila por fila") y guardamos el resultado
rfm["Segmento"] = rfm.apply(asignar_segmento, axis=1)

# Resumimos por segmento: cuántos clientes hay y sus promedios de R, F y M
resumen = rfm.groupby("Segmento").agg(
    Clientes=("Customer ID", "count"),
    Recencia_prom=("Recencia", "mean"),
    Frecuencia_prom=("Frecuencia", "mean"),
    Monto_prom=("Monto", "mean"),
    Monto_total=("Monto", "sum")
)

# Calculamos qué porcentaje del ingreso total aporta cada segmento
resumen["% del ingreso"] = resumen["Monto_total"] / resumen["Monto_total"].sum() * 100

# Calculamos qué porcentaje de los clientes representa cada segmento
resumen["% de clientes"] = resumen["Clientes"] / resumen["Clientes"].sum() * 100

# Mostramos la tabla ordenada de mayor a menor ingreso
resumen.sort_values("Monto_total", ascending=False)

## Bloque 8: Guardar los resultados
Exportamos los archivos que usa la app de Streamlit. El CSV completo de ventas pesa ~77 MB y **no cabe en GitHub** (límite de 25 MB desde la web), así que también guardamos una versión comprimida y liviana (`.csv.gz`, ~9 MB) con solo las columnas que necesita la app.

In [ ]:
# Guardamos la tabla de clientes con su RFM y segmento; index=False evita una columna de números sobrante
rfm.to_csv("clientes_rfm.csv", index=False)

# Guardamos el dataset limpio completo (archivo grande, útil para análisis propios)
limpio.to_csv("ventas_limpias.csv", index=False)

# Elegimos solo las columnas que necesita la app y redondeamos el ingreso a 2 decimales
ventas_app = limpio[["Invoice", "Description", "InvoiceDate", "Customer ID", "Country", "Ingreso"]].copy()
ventas_app["Ingreso"] = ventas_app["Ingreso"].round(2)

# Guardamos la versión liviana comprimida (pandas la comprime sola por la extensión .gz)
ventas_app.to_csv("ventas_limpias.csv.gz", index=False, compression="gzip")

# Confirmamos que todo salió bien
print("Archivos guardados: clientes_rfm.csv, ventas_limpias.csv y ventas_limpias.csv.gz")

# Descargamos a tu computador solo los dos archivos que usa la app (el CSV grande tardaría mucho)
from google.colab import files
files.download("clientes_rfm.csv")
files.download("ventas_limpias.csv.gz")

## 3 hallazgos de negocio

1. **Pocos clientes generan casi todo el dinero.** Los *Campeones* son el **22.5 %** de los clientes pero aportan el **70 %** del ingreso (£12.2 M de £17.5 M). Perder a uno de ellos duele mucho más que perder a un cliente ocasional: conviene un programa de atención preferente.

2. **Los clientes "En Riesgo" son la mejor oportunidad de recuperación.** Son **972** clientes que compraron en promedio **£1,885** cada uno (**6.6 veces** más que un cliente Perdido, £285), pero llevan unos **374 días** sin volver. Reactivarlos cuesta mucho menos que conseguir clientes nuevos.

3. **El reto es lograr la segunda compra.** El **27.6 %** de los clientes compró una sola vez, y apenas representa el **3.2 %** del ingreso. En el segmento *Perdidos*, el **80 %** nunca repitió; en *Potenciales*, casi la mitad (48 %) todavía tiene solo una compra. Una estrategia de bienvenida y recompra es clave.

*Las cifras salen de las tablas de este notebook; si cambias la limpieza, vuelve a ejecutarlo para actualizarlas.*